# Hướng dẫn chạy dự án trên Kaggle / Google Colab

Notebook này đóng vai trò như một **"bảng điều khiển"** (remote control) để chạy các script `.py` của dự án trên các máy chủ có GPU/TPU mạnh mẽ của Kaggle hoặc Google Colab.

Thay vì viết code trực tiếp vào các cell (như trước đây), chúng ta sẽ sử dụng dấu `!` để chạy các lệnh terminal. Điều này giúp tận dụng sức mạnh tính toán của đám mây trong khi vẫn giữ cấu trúc mã nguồn dạng mô-đun gọn gàng.

## 1. Chuẩn bị môi trường (Setup Environment)

**Lưu ý quan trọng:** Bạn cần đảm bảo mã nguồn (`src/text-sumarization`) đã được tải lên môi trường này.
- **Google Colab:** Tải thư mục này lên Google Drive và mount ổ đĩa.
- **Kaggle:** Nén (zip) thư mục này lại, tải lên dưới dạng *Kaggle Dataset* và copy vào thư mục `/kaggle/working/`.

Chạy cell dưới đây để thiết lập môi trường. (Hãy bỏ comment (`#`) các dòng phù hợp với môi trường của bạn).

In [ ]:
# =========================================================
# DÀNH CHO GOOGLE COLAB:
# =========================================================
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Đường_dẫn_tới/text-sumarization

# =========================================================
# DÀNH CHO KAGGLE:
# =========================================================
# !cp -r /kaggle/input/ten-dataset-cua-ban/text-sumarization /kaggle/working/
# %cd /kaggle/working/text-sumarization

# Cài đặt các thư viện (dependencies) cần thiết
!pip install -e .

## 2. Huấn luyện mô hình (Training)

Sử dụng lệnh `python scripts/train.py` để bắt đầu quá trình huấn luyện.
HuggingFace Trainer (được sử dụng ngầm bên trong script) sẽ **tự động phát hiện và sử dụng GPU hoặc TPU** nếu có sẵn trên máy ảo này.

Bạn có thể ghi đè các thông số (như epoch, batch size) trực tiếp từ dòng lệnh mà không cần phải mở file config ra sửa.

In [ ]:
# Chạy file train.py bằng dấu chấm than (!)
# Lưu output vào một thư mục dễ tìm
!python scripts/train.py \
    --config configs/vit5_base.yaml \
    --output-dir ./outputs/vit5_experiment \
    --epochs 3 \
    --batch-size 4

## 3. Đánh giá mô hình (Evaluation)

Sau khi quá trình huấn luyện kết thúc, bạn có thể chạy lệnh đánh giá để tính toán các điểm số ROUGE trên tập dữ liệu validation.

In [ ]:
!python scripts/evaluate.py \
    --model ./outputs/vit5_experiment/best \
    --config configs/vit5_base.yaml

## 4. Dự đoán thử nghiệm (Inference)

Bạn có thể sử dụng checkpoint mô hình vừa huấn luyện tốt nhất để tóm tắt một đoạn văn bản thực tế.

In [ ]:
text_to_summarize = """
Trong bối cảnh nền kinh tế toàn cầu đang đối mặt với nhiều thách thức, 
xuất khẩu của Việt Nam trong quý 1 vẫn ghi nhận những tín hiệu tích cực. 
Đặc biệt, nhóm hàng nông sản và điện tử đóng góp phần lớn vào tổng kim ngạch xuất khẩu. 
Chính phủ tiếp tục đưa ra các chính sách hỗ trợ doanh nghiệp nhằm thúc đẩy sản xuất và mở rộng thị trường.
"""

# Lưu đoạn văn bản ra một file tạm
with open("sample_article.txt", "w", encoding="utf-8") as f:
    f.write(text_to_summarize)

# Gọi script predict.py để tóm tắt file này
!python scripts/predict.py \
    --model ./outputs/vit5_experiment/best \
    --file sample_article.txt

## 5. Lưu trữ và Tải kết quả về (Save & Download Results)

**Cảnh báo:** Colab và Kaggle sẽ **xóa toàn bộ dữ liệu** khi phiên làm việc (session) kết thúc hoặc bị ngắt kết nối. 

Do đó, sau khi huấn luyện xong, bạn bắt buộc phải nén (zip) thư mục chứa checkpoint kết quả và tải về máy tính cá nhân (hoặc copy sang Google Drive).

In [ ]:
# Nén toàn bộ thư mục outputs thành file zip
!zip -r my_model_results.zip ./outputs/vit5_experiment/

# Đoạn code Python này giúp tự động mở hộp thoại tải file xuống (chỉ hoạt động trên Colab)
try:
    from google.colab import files
    files.download('my_model_results.zip')
    print("Đang tải file xuống...")
except ImportError:
    print("Nếu bạn đang dùng Kaggle, hãy tìm file 'my_model_results.zip' ở thanh sidebar bên phải (mục Output) để tải về.")